# 路由与请求参数

学习目标：从路径、查询字符串、请求头和 Cookie 读取参数，设置输入约束，并检查参数来源与路由匹配顺序。

前置知识：HTTP 请求、Python 函数参数、基本类型标注与 FastAPI 路由。

适用版本：Python 3.12、FastAPI 0.141.1、Pydantic v2。

环境准备：[FastAPI 环境与运行说明](README.md)。

工作目录：content/Web与应用开发/FastAPI。按顺序执行本篇单元；本章使用应用内请求，不启动网络服务，也不需要配套脚本。

署名：CMYK Labs（cmyk-labs）；原创教程与代码采用 CC BY-NC-SA 4.0。

## 1 从路径取得记录编号

要读取编号为 7 的记录，可以请求 /records/7。路由中的 {record_id} 是占位符，record_id 表示记录编号；FastAPI 把匹配到的文本交给同名函数参数，并按照 int 标注转换和校验。

下面用 TestClient 在应用内发请求。get() 发送 GET 请求，json() 读取 JSON 响应；with 块结束时关闭客户端。

In [1]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI()


@app.get("/records/{record_id}")
def read_record(record_id: int):
    return {"record_id": record_id}


with TestClient(app) as client:
    response = client.get("/records/7")

assert response.status_code == 200
assert response.json() == {"record_id": 7}
print(response.json())  # 预期：{'record_id': 7}。
# 返回的是整数 7，路径文本已经按 int 标注转换。

{'record_id': 7}


C:\Users\ZHUANG\miniconda3\envs\hands-on-computing\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


如果路径文本不能转为整数，FastAPI 返回请求校验错误。错误中的 loc 说明参数位置，type 说明错误类型；这种请求不会进入路由函数。

In [2]:
with TestClient(app) as client:
    response = client.get("/records/abc")

error = response.json()["detail"][0]
assert response.status_code == 422
assert error["loc"] == ["path", "record_id"]
print(response.status_code, error["loc"], error["type"])  # 预期：422 ['path', 'record_id'] int_parsing。
# path 表示错误来自路径；record_id 指出失败的参数。

422 ['path', 'record_id'] int_parsing


## 2 固定路径放在动态路径之前

如果还需要 /records/latest 表示最新记录，它也能匹配 /records/{record_id} 的形状。FastAPI 按注册顺序匹配路由，因此应先声明固定路径。

下面新建一个小应用演示正确顺序，避免前面已经注册的路由干扰观察。若先注册动态路径，latest 会被当作编号处理；整数校验失败后，不会回头寻找后面的固定路由。

In [3]:
ordered_app = FastAPI()


@ordered_app.get("/records/latest")
def latest_record():
    return {"record_id": 3, "route": "latest"}


@ordered_app.get("/records/{record_id}")
def numbered_record(record_id: int):
    return {"record_id": record_id, "route": "numbered"}


with TestClient(ordered_app) as client:
    latest = client.get("/records/latest")
    numbered = client.get("/records/7")

assert latest.json()["route"] == "latest"
assert numbered.json()["route"] == "numbered"
print(latest.json(), numbered.json())  # 预期：{'record_id': 3, 'route': 'latest'} {'record_id': 7, 'route': 'numbered'}。
# 两个请求分别进入固定路由与动态路由。

{'record_id': 3, 'route': 'latest'} {'record_id': 7, 'route': 'numbered'}


## 3 查询参数的必填与默认值

查询字符串适合表达筛选条件，例如 /search?topic=Python。未出现在路径中的简单类型参数，默认从查询字符串取得。

topic 没有默认值，必须提供；limit 表示最多返回几条记录，默认 2，可以省略。客户端的 params 参数负责构造查询字符串。这里先返回收到的参数，便于观察取值，不引入实际搜索代码。

In [4]:
@app.get("/search")
def search_records(topic: str, limit: int = 2):
    return {"topic": topic, "limit": limit}


with TestClient(app) as client:
    defaulted = client.get("/search", params={"topic": "Python"})
    supplied = client.get("/search", params={"topic": "Python", "limit": "3"})

assert defaulted.json() == {"topic": "Python", "limit": 2}
assert supplied.json()["limit"] == 3
print(defaulted.json(), supplied.json())  # 预期：{'topic': 'Python', 'limit': 2} {'topic': 'Python', 'limit': 3}。
# 省略 limit 使用默认值；提供文本 "3" 则按 int 转换。

{'topic': 'Python', 'limit': 2} {'topic': 'Python', 'limit': 3}


缺失参数与无法转换的参数，是两种不同的失败。前者没有提供必需值，后者提供了值但不满足类型要求。

In [5]:
with TestClient(app) as client:
    missing = client.get("/search")
    invalid = client.get("/search", params={"topic": "Python", "limit": "many"})

assert missing.status_code == invalid.status_code == 422
for response in [missing, invalid]:
    error = response.json()["detail"][0]
    print(error["loc"], error["type"])  # 预期：依次为 ['query', 'topic'] missing、['query', 'limit'] int_parsing。
# 分别定位到缺失的 topic 和无法转换的 limit。

['query', 'topic'] missing
['query', 'limit'] int_parsing


## 4 用 Annotated、Path 与 Query 添加约束

只检查整数类型，还不能拒绝负编号或过大的数量。Annotated 可以把类型和额外元数据写在一起：int 描述类型，Path 与 Query 明确参数来源并添加约束。

下面要求 record_id 大于等于 1、limit 在 1～5 之间。ge 表示大于等于，le 表示小于等于。使用 Annotated 时，默认值放在函数参数的等号后，不再同时写进 Query(default=...)；路径参数始终必需。

In [6]:
from typing import Annotated

from fastapi import Path, Query


@app.get("/bounded-records/{record_id}")
def bounded_records(
    record_id: Annotated[int, Path(ge=1)],
    limit: Annotated[int, Query(ge=1, le=5)] = 2,
):
    return {"record_id": record_id, "limit": limit}


with TestClient(app) as client:
    response = client.get("/bounded-records/1", params={"limit": 5})

assert response.json() == {"record_id": 1, "limit": 5}
print(response.json())  # 预期：{'record_id': 1, 'limit': 5}。
# 两个数值都处在允许的边界上。

{'record_id': 1, 'limit': 5}


约束需要同时检查允许的边界和超出边界的值。错误位置仍能区分路径与查询字符串，便于客户端修正对应输入。

In [7]:
with TestClient(app) as client:
    for path, limit in [("/bounded-records/0", 2), ("/bounded-records/1", 6)]:
        response = client.get(path, params={"limit": limit})
        assert response.status_code == 422
        error = response.json()["detail"][0]
        # 预期：record_id 位于 path，错误为 greater_than_equal；limit 位于 query，错误为 less_than_equal。
        print(error["loc"], error["type"])
# 第一个请求的编号过小，第二个请求的数量过大。

['path', 'record_id']

 greater_than_equal
['query', 'limit'] less_than_equal


Query 也能限制字符串长度。下面的 q 表示可选搜索词，min_length 与 max_length 要求提供时长度为 2～10 个字符。

q: str | None 允许 None，等号后的 None 则使参数可以省略。省略参数与传入空字符串不同：空字符串仍需要通过长度校验。

In [8]:
@app.get("/keywords")
def read_keywords(
    q: Annotated[str | None, Query(min_length=2, max_length=10)] = None,
):
    return {"q": q}


with TestClient(app) as client:
    absent = client.get("/keywords")
    supplied = client.get("/keywords", params={"q": "API"})
    empty = client.get("/keywords", params={"q": ""})

assert absent.json() == {"q": None}
assert supplied.json() == {"q": "API"}
assert empty.status_code == 422
print(absent.json(), supplied.json(), empty.status_code)  # 预期：{'q': None} {'q': 'API'} 422。
# 省略 q 合法，显式提供空字符串不满足最小长度。

{'q': None}

 {'q': 'API'} 422


## 5 明确从请求头读取参数

Header 声明从请求头取值。默认情况下，Python 参数名 x_client 的下划线会转换成连字符，对应请求头 X-Client；请求头名称不区分大小写。

如果没有 Header 声明，简单字符串参数通常会被解释为查询参数。参数写在哪个来源，比名称看起来像什么更重要。

In [9]:
from fastapi import Header


@app.get("/client-info")
def client_info(x_client: Annotated[str, Header()]):
    return {"client": x_client}


with TestClient(app) as client:
    correct = client.get("/client-info", headers={"X-Client": "notebook"})
    wrong_source = client.get("/client-info", params={"x_client": "notebook"})

assert correct.json() == {"client": "notebook"}
assert wrong_source.status_code == 422
print(correct.json(), wrong_source.json()["detail"][0]["loc"])  # 预期：{'client': 'notebook'} ['header', 'x-client']。
# 查询字符串中的 x_client 不能替代必需的 X-Client 请求头。

{'client': 'notebook'} ['header', 'x-client']


## 6 从 Cookie 读取偏好值

Cookie 明确声明从 Cookie 读取参数。下面用 theme 表示界面主题，省略时采用 light。它只是偏好值，不用于证明用户身份。

通过客户端的 Cookie 容器准备输入，就能在 Notebook 中观察读取行为。只写 theme: str，并不会自动让 FastAPI 从 Cookie 取值。

In [10]:
from fastapi import Cookie


@app.get("/preferences")
def read_preferences(theme: Annotated[str, Cookie()] = "light"):
    return {"theme": theme}


with TestClient(app) as client:
    defaulted = client.get("/preferences")
    client.cookies.set("theme", "dark")
    supplied = client.get("/preferences")

assert defaulted.json() == {"theme": "light"}
assert supplied.json() == {"theme": "dark"}
print(defaulted.json(), supplied.json())  # 预期：{'theme': 'light'} {'theme': 'dark'}。
# 第一次没有 Cookie，第二次从客户端 Cookie 容器发送 theme=dark。

{'theme': 'light'}

 {'theme': 'dark'}


## 7 核对参数来源声明

OpenAPI 是描述接口的结构化文档。FastAPI 会把参数的位置、类型、约束和必填条件写入其中；parameter.in 表示来源，required 表示是否必填。下面用本篇已经定义好的路由核对声明。

| 名称 | 中文名称／含义 | 本篇参数 |
| --- | --- | --- |
| Path | 路径参数 | record_id |
| Query | 查询参数 | limit、q |
| Header | 请求头参数 | x_client |
| Cookie | Cookie 参数 | theme |

In [11]:
schema = app.openapi()
paths = ["/bounded-records/{record_id}", "/client-info", "/preferences"]

for path in paths:
    parameters = schema["paths"][path]["get"]["parameters"]
    # 预期：依次列出：record_id/path/必填、limit/query/可选；x-client/header/必填；theme/cookie/可选。
    print(path, [(item["name"], item["in"], item["required"]) for item in parameters])

header_parameter = schema["paths"]["/client-info"]["get"]["parameters"][0]
cookie_parameter = schema["paths"]["/preferences"]["get"]["parameters"][0]
assert header_parameter["in"] == "header" and header_parameter["required"]
assert cookie_parameter["in"] == "cookie" and not cookie_parameter["required"]
# 文档声明与请求测试分别检查；文档正确不替代实际取值检查。

/bounded-records/{record_id} [('record_id', 'path', True), ('limit', 'query', False)]
/client-info [('x-client', 'header', True)]
/preferences [('theme', 'cookie', False)]


## 本章小结

（1）路径按注册顺序匹配；固定路径与动态路径可能冲突时，把固定路径放在前面。

（2）默认值决定参数能否省略，类型和约束决定提供的值是否合法。None 与空字符串不能混为一谈。

（3）Path、Query、Header 和 Cookie 明确参数来源；校验错误与 OpenAPI 都能帮助检查声明是否符合预期。

自查：为什么把 X-Client 的值写进查询字符串后，接口仍报告缺少请求头？

## 练习

（1）为 /search 增加 offset 查询参数，默认 0，且不得小于 0。分别请求省略、0、-1 三种输入，断言前两种成功、最后一种返回 422。

（2）新建一个小应用，同时提供 /records/count 和 /records/{record_id}。让 count 返回固定数量 3，检查固定路径与整数路径分别进入预期函数。交换注册顺序后再次运行，比较 /records/count 的结果。

（3）新增一个同时接收必填 X-Trace 请求头和可选 theme Cookie 的接口。用两种输入测试：正确提供请求头与 Cookie；仅在查询字符串提供 x_trace。断言前者返回对应值，后者返回 422。

提示：修改应用定义后重新运行相关定义，必要时从空内核顺序执行；第二题每次比较都创建新的应用，避免旧路由留下干扰。

## 参考与引用来源

来源核查日期：2026-09-15。以下为官方或第一方资料；示例为本课程原创。

（1）**FastAPI 官方文档**：[Path Parameters](https://fastapi.tiangolo.com/tutorial/path-params/)，定位 Path parameters with types、Data conversion、Data validation、Order matters，支持第 1～2 节；[Query Parameters](https://fastapi.tiangolo.com/tutorial/query-params/)，定位 Defaults 与 Required query parameters，支持第 3 节；[Query Parameters and String Validations](https://fastapi.tiangolo.com/tutorial/query-params-str-validations/)，定位 Annotated、Query as the default value or in Annotated、Add more validations，支持第 4 节；[Path Parameters and Numeric Validations](https://fastapi.tiangolo.com/tutorial/path-params-numeric-validations/)，定位 Number validations，支持第 4 节；[Header Parameters](https://fastapi.tiangolo.com/tutorial/header-params/#automatic-conversion)，定位声明来源与 Automatic conversion，支持第 5 节；[Cookie Parameters](https://fastapi.tiangolo.com/tutorial/cookie-params/)，定位 Declare Cookie parameters，支持第 6 节；上述页面的 Documentation 与参数声明部分支持第 7 节；[Testing](https://fastapi.tiangolo.com/tutorial/testing/#using-testclient)，支持应用内请求方式。

（2）**Starlette 官方文档**：[TestClient](https://starlette.dev/testclient/)，定位客户端请求与上下文管理器；本课程锁定的 Starlette 1.6.0 仍支持 HTTPX。

（3）**GitHub 上的第一方源码**：[Starlette 1.6.0 testclient.py](https://github.com/Kludex/starlette/blob/1.6.0/starlette/testclient.py#L29)，定位 HTTPX 导入兼容分支。

（4）**OpenAPI 官方规范**：[OpenAPI 3.1.0 §4.8.12 Parameter Object](https://spec.openapis.org/oas/v3.1.0.html#parameter-object)，定位 Parameter Locations 与 Fixed Fields，支持第 7 节中的 in 与 required。